## Programming for Data Science 
### Project Notebook: "Where should I live?" 
#### Group Members:
- Afonso Fernandes / 20241710
- Lourenço Lima / 20241711
- Pedro Jorge / 20241819
- David Morais / 20241759
## Project Repository
GitHub Repository:  
https://github.com/afonsolince06/-Where-should-I-live-PDS-Project


### **Introduction**

In this part of the project, the goal is to build an interactive map of Europe that allows users to explore key information about major European cities. The task combines web scraping, data integration, and geospatial visualization to create an informative and interactive tool.

To accomplish this, we will:

-Scrape the geographical coordinates of each city directly from the Wikipedia Main Page, ensuring accuracy and consistency with the provided dataset.

-Match the scraped coordinates with the dataset entries so that each city is correctly assigned to its corresponding country, population, average salary, and cost of living.

-Use the cleaned and enriched dataset to construct an interactive map of Europe, where each city can be clicked or hovered over to display its relevant information.

By the end of this section, we will have a fully functional map that visually represents European cities and provides meaningful insights through an intuitive interface. This builds on the skills developed earlier in the project and introduces new concepts in geospatial data handling and visualization.

### **Importing the necessary libraries for webscraping and map building.**

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### **Importing the already cleaned dataset**

In [25]:
city_data = pd.read_csv('city_data_cleaned.csv') # Load the cleaned city data
city_data_coords = city_data.copy() # Create a copy to add coordinates
# Fill the coordinate columns with NaN values
city_data_coords['Latitude'] = np.nan
city_data_coords['Longitude'] = np.nan

### **5. Web Scrapping**
In this section, Selenium is used to automatically retrieve the geographic coordinates (latitude and longitude) of European cities from the cleaned dataset `city_data_coords` using information available on Wikipedia. The process begins with the implementation of the function `get_city_coordinates`, which opens a web browser and navigates to the Wikipedia main page. For each city in the dataset, a search is performed—optionally including the country name to improve accuracy—followed by navigation to the corresponding city article, where the geographic coordinates are extracted.

The retrieved coordinates are subsequently stored in the dataset through the function `fill_dataset_coords`.

In [26]:
wikipedia_url = "https://en.wikipedia.org/wiki/Main_Page" # Wikipedia main page URL


def get_city_coordinates(city_name):
    """
    Starts on the Wikipedia main page and searches for the given city name.

    Parameters:
    city_name (str): The name of the city to search for.

    Returns:
    The city page URL.
    """

    if city_name == 'Rotterdam': # Special case for Rotterdam
        try:
            # Navigate to Wikipedia main page
            driver.get(wikipedia_url)

            # Find the search input box and enter the city name
            search_input = WebDriverWait(driver, 15).until(EC.element_to_be_clickable((By.ID, "searchInput")))
            search_input.send_keys(city_name)
            search_input.send_keys(Keys.RETURN)

        
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, "firstHeading"))) # Wait for page to load
            lat = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//span[@class='latitude']")))
            latitude = driver.execute_script("return arguments[0].textContent;", lat)
            long = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, "//span[@class='longitude']")))
            longitude = driver.execute_script("return arguments[0].textContent;", long)
            return (latitude.strip(), longitude.strip())
        
        finally:
            driver.quit() # Close the browser

    else: # General case for other cities
        # Set up the Selenium WebDriver 
        options = webdriver.ChromeOptions()
        driver = webdriver.Chrome(options=options)
        driver.maximize_window()
        
        try:
            # Navigate to Wikipedia main page
            driver.get(wikipedia_url)
            # Find the search input box and enter the city name
            search_input = WebDriverWait(driver, 15).until(EC.element_to_be_clickable((By.ID, "searchInput")))
            search_input.send_keys(city_name)
            search_input.send_keys(Keys.RETURN)
          
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.ID, "firstHeading"))) # Wait for page to load
            city_latitude = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "latitude")))
            city_longitude = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "longitude")))
            return (city_latitude.text, city_longitude.text)
            
        finally:
            driver.quit() # Close the browser
    


def fill_dataset_coords(df):
    """
    Fills the dataset with latitude and longitude coordinates for each city.
    Parameters:
    df (DataFrame): The DataFrame containing city data.
    Returns:
    The updated DataFrame with latitude and longitude columns filled.
    """

    for city in df['City']:
        try: # Attempt to get coordinates using "City, Country" format first
            df.loc[df['City'] == city, ['Latitude', 'Longitude']] = get_city_coordinates(f"{city}, {df.loc[city_data_coords['City'] == city, 'Country'].values[0]}")
        except Exception: # If that fails, try just the city name
            df.loc[df['City'] == city, ['Latitude', 'Longitude']] = get_city_coordinates(city)
    return df





# Filling the dataset with coordinates
fill_dataset_coords(city_data_coords)


display(city_data_coords.tail(40)) # Checking the last 40 rows to check if the function worked for Rotterdam as it was the only city we had issues with
city_data_coords.loc[city_data_coords['City'] == 'Rotterdam', ['Latitude', 'Longitude']] = ('51°55′N', '4°29′E') # As we were not able to scrapte the coordinates for Rotterdam, we will fill it manually just so we have it in the map
display(city_data_coords.loc[city_data_coords['City'] == 'Rotterdam', ['City','Latitude', 'Longitude']]) # Checking if Rotterdam coordinates were filled correctly

C:\Users\ASUS\AppData\Local\Temp\ipykernel_19136\4113309029.py:71: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '47°48′00″N' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.

C:\Users\ASUS\AppData\Local\Temp\ipykernel_19136\4113309029.py:71: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '13°02′42″E' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.



,Country,City,Population Density,Population,Working Age Population,Youth Dependency Ratio,Unemployment Rate,GDP per Capita,Days of very strong heat stress,Main Spoken Languages,Average Monthly Salary,Average Rent Price,Average Cost of Living,Last Data Update,Individual Languages,Spendable Income,Latitude,Longitude
44,Italy,Venice,852.0,557748,347781.0,19.3,6.1,39681.0,6,Italian,1400,1150,1500,2023-03-23,['Italian'],-100,45°26′15″N,12°20′9″E
45,Latvia,Riga,152.0,930245,594604.0,25.6,5.3,39537.0,0,"Latvian, Russian",1300,600,1250,2023-12-07,"['Latvian', 'Russian']",50,56°56′56″N,24°6′23″E
46,Luxembourg,Luxembourg,236.0,610825,424824.0,23.1,5.6,112143.0,2,"Luxembourgish, French, German, English",4200,2100,3300,2024-11-29,"['Luxembourgish', 'French', 'German', 'English']",900,49°36′42″N,6°7′55″E
47,Malta,Malta,1845.0,456490,309610.0,20.2,3.7,44781.0,0,"Maltese, English",1450,1000,1400,2024-02-28,"['Maltese', 'English']",50,35°54′N,14°31′E
48,Netherlands,Amsterdam,864.0,2862590,1913803.0,24.0,3.3,73065.0,2,"Dutch, English",3750,1150,2550,2024-07-21,"['Dutch', 'English']",1200,52°22′22″N,04°53′37″E
49,Netherlands,Eindhoven,551.0,760059,495081.0,23.2,3.3,62660.0,3,"Dutch, English",3100,1100,1900,2023-06-20,"['Dutch', 'English']",1200,51°26′N,5°29′E
50,Netherlands,Rotterdam,864.0,1860582,1213524.0,25.2,4.1,54780.0,2,"Dutch, English",3100,1350,2100,2023-10-11,"['Dutch', 'English']",1000,,
51,Netherlands,The Hague,2646.0,1103137,733794.0,25.1,4.2,54976.0,1,"Dutch, English",3500,1250,2000,2023-09-17,"['Dutch', 'English']",1500,52°04′48″N,04°18′36″E
52,Netherlands,Utrecht,924.0,898712,592246.0,26.2,3.0,67062.0,3,"Dutch, English",3200,1300,2000,2024-05-26,"['Dutch', 'English']",1200,52°05′27″N,05°07′18″E
53,Norway,Bergen,126.0,419854,278139.0,27.2,4.6,51095.0,0,"Norwegian, English",3500,1200,2100,2023-04-30,"['Norwegian', 'English']",1400,60°23′22″N,5°19′48″E


,City,Latitude,Longitude
50,Rotterdam,51°55′N,4°29′E


### **Exporting the final DataFrame with the scraped coordinates (in DMS format)**

In [27]:
city_data_coords.to_csv('city_data_with_coords.csv', index=False) # Saving the dataset with coordinates to a CSV file

# Safety net in case we need to reload the DataFrame with coordinates after deleting them by accident
city_data_coords = pd.read_csv('city_data_with_coords.csv')

### **6. Interactive Map**

This section is where we build the interactive map with all the cities in the provided dataset and the hover information that was asked for in the guidelines.

Since the coordinates obtained from Wikipedia are provided in DMS (Degrees, Minutes, Seconds) format, an additional function, `dms_to_decimal`, is applied to convert them into decimal degrees. These processed coordinates are then used as input for the construction of an interactive map, allowing for a clear geographical representation of the analyzed cities.

In [ ]:
# Iterative Map Building and Visualization

def dms_to_decimal(dms_coords):
    """
    Converts the coordinates from the DMS (Degrees, Minutes, Seconds) format that they were scraped in to a decimal format to be used in the map visualization.
    
    Parameters:
    dms_coords (str): The coordinates in DMS format.
    
    Returns:
    The coordinates in decimal format.
    """
    dms_parts = re.split('[°′″NEW]', dms_coords) # Split the DMS string into its components and taking care of N, E, W, S characters
    degrees = float(dms_parts[0]) # Degrees part is always present and it's the first element
    minutes = float(dms_parts[1]) if len(dms_parts) > 1 and dms_parts[1] else 0 # Minutes part may be missing, it's the second element
    seconds = float(dms_parts[2]) if len(dms_parts) > 2 and dms_parts[2] else 0 # Seconds part may be missing, it's the third element

    decimal_coords = degrees + (minutes / 60) + (seconds / 3600) # Formula to convert DMS to decimal

    if 'S' in dms_coords or 'W' in dms_coords:
        decimal_coords = -decimal_coords # Southern and western hemispheres are negative

    return decimal_coords

city_data_coords['Latitude'] = city_data_coords['Latitude'].apply(dms_to_decimal)
city_data_coords['Longitude'] = city_data_coords['Longitude'].apply(dms_to_decimal)

# Drop rows with missing coordinates so the map visualization works properly
map_df = city_data_coords.dropna(subset=['Latitude', 'Longitude'])

# Create the scatter map
fig_map = px.scatter_map(
        map_df,
        lat="Latitude",
        lon="Longitude",
        hover_name="City",
        hover_data={
            "Latitude": False, "Longitude": False,
            "Country": True,
            "Population": ':,.0f', 
            "Average Monthly Salary": ':,.0f €', 
            "Average Cost of Living": ':,.0f €'  
        },
        color="Average Monthly Salary", 
        size="Population", 
        size_max=25,             
        color_continuous_scale="Turbo",
        zoom=3, 
        height=700,
        title="Iterative Map of Cities with Population, Average Monthly Salary, and Cost of Living"
    )

# Update layout for better visualization
fig_map.update_layout(mapbox_style="open-street-map")
fig_map.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig_map.show()